# Text Generation with a From-Scratch RNN

This notebook trains a vanilla recurrent neural network — implemented from
scratch in NumPy (see [`rnn_scratch.py`](rnn_scratch.py)) — to model text and
generate new sequences. It works at two levels:

1. **Character level** — one-hot characters in, predict the next character.
2. **Word level** — pre-trained word embeddings in (Word2Vec, GloVe-style
   Google-News word2vec, FastText, BERT), predict the next word.

**Pipeline:** raw text → vocabulary → training sequences → train RNN →
predict the next token / generate a sequence.

| Section | Contents |
|---------|----------|
| 1 | Imports |
| 2 | Data |
| 3 | Character-level text generation |
| 4 | Word-level prediction (Word2Vec · GloVe · FastText · BERT) |

## 1. Imports

In [1]:
import importlib

import numpy as np

import rnn_scratch
import utils

# Reload the local modules so edits to the .py files are picked up without
# having to restart the kernel.
importlib.reload(rnn_scratch)
importlib.reload(utils)

from rnn_scratch import RNN
from utils import generate_dataset, predict_next, generate

## 2. Data

A small block of text to learn from. With a corpus this size the model can only
learn surface character statistics, but it is enough to demonstrate the RNN.

In [2]:
sentences = [
'Machine Learning (ML) is a branch of Artificial Intelligence (AI)',
'that enables computers to learn from data and improve their performance',
'without being explicitly programmed. ML algorithms identify patterns in',
'historical data and use those patterns to make predictions or decisions on new data.',
'Machine learning is widely used in various applications such as recommendation systems,',
'image recognition, speech processing, fraud detection, healthcare diagnostics,',
'and autonomous vehicles. The main types of machine learning are supervised learning,',
'unsupervised learning, and reinforcement learning. As the amount of available data continues to grow,',
'machine learning plays an increasingly important role in helping organizations automate tasks,',
'gain insights, and make data-driven decisions.'
]
sentence_words = [s.split() for s in sentences]
words = [w for s in sentence_words for w in s]
text = " ".join(words)

## 3. Character-level Text Generation

We slide a window of length `T_x` over the text. Each input sequence `X` is a
chunk of characters and the target `Y` is the same chunk shifted one character
to the left, so the model learns to predict the *next* character at every step.

The arrays are one-hot encoded into shape `(vocab_size, m, T_x)`:

- `vocab_size` — one-hot dimension of each character
- `m`          — number of training sequences
- `T_x`        — time steps per sequence

In [3]:
# Build one-hot character sequences (word_vectors is unused in char mode).
X, Y, char_to_index, index_to_char = generate_dataset(
    text, T_x=10, is_char=True, word_vectors=[], seq_length=25
)
print(f"X shape: {X.shape}   Y shape: {Y.shape}   (vocab_size, m, T_x)")

# Train a vanilla RNN to predict the next character.
text_rnn = RNN(X, Y, n_a=100, learning_rate=0.001, iterations=1000)
text_rnn.train()

X shape: (35, 754, 10)   Y shape: (35, 754, 10)   (vocab_size, m, T_x)
Iteration: 0, Loss: 35.554020357057496
Iteration: 100, Loss: 22.38848844124454
Iteration: 200, Loss: 12.78796470456537
Iteration: 300, Loss: 7.612424147532962
Iteration: 400, Loss: 5.621841557743679
Iteration: 500, Loss: 4.825149647416697
Iteration: 600, Loss: 4.409665462361317
Iteration: 700, Loss: 4.181462646903308
Iteration: 800, Loss: 4.063006708918638
Iteration: 900, Loss: 3.9958509506850244


In [4]:
# single next-character prediction
print("next char after 'M':", predict_next(text_rnn, char_to_index, index_to_char,seed_word ='M', is_char=True))

# generate a sequence from the seed character
print()
print(generate(text_rnn, char_to_index, index_to_char, seed_word='A', num_words=100, is_char=True))

next char after 'M': L

Artificial Intelligence (AI) that enables computers to learn from data and use those patterns in hist


## 4. Word-level Prediction

Instead of one-hot characters, each input token is now a dense **word
embedding**. The targets stay one-hot over the vocabulary, so the RNN runs in
`task="classification"` mode and predicts the next word. The four subsections
below feed the same corpus through different embeddings and compare the result.

### 4.1 Word2Vec embeddings

Train a Word2Vec model on our own sentences (skip-gram) and use its learned
vectors as the RNN's input features.

In [5]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
word_vectors = w2v_model.wv

In [6]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(words, T_x=5, is_char=False, word_vectors=word_vectors)
print(f"input_seqs shape: {input_seqs.shape} target_seqs shape: {target_seqs.shape} (vector_size, m, T_x)")

w2c_rnn = RNN(input_seqs, target_seqs, n_a=100, learning_rate=0.01, iterations=2000,
              task="classification")
w2c_rnn.train()

input_seqs shape: (100, 102, 5) target_seqs shape: (88, 102, 5) (vector_size, m, T_x)
Iteration: 0, Loss: 22.386632874882427
Iteration: 100, Loss: 21.750017226733572
Iteration: 200, Loss: 21.72706863160117
Iteration: 300, Loss: 21.696197133661368
Iteration: 400, Loss: 21.61459590304818
Iteration: 500, Loss: 20.582624859551583
Iteration: 600, Loss: 17.193594456436344
Iteration: 700, Loss: 9.747231439291784
Iteration: 800, Loss: 4.965075191038878
Iteration: 900, Loss: 3.456034109026806
Iteration: 1000, Loss: 3.13862867989043
Iteration: 1100, Loss: 2.4434489514427122
Iteration: 1200, Loss: 2.0713121387312
Iteration: 1300, Loss: 1.8178063128958708
Iteration: 1400, Loss: 1.5877290524744636
Iteration: 1500, Loss: 1.3795988000950434
Iteration: 1600, Loss: 1.5920201776891227
Iteration: 1700, Loss: 1.2411113473497526
Iteration: 1800, Loss: 0.9989196731779609
Iteration: 1900, Loss: 1.0103313268632115


In [7]:
# single next-word prediction (most likely word)
print("next word after 'Machine':", predict_next(w2c_rnn, word_vectors, index_to_vocab, 'Machine'))

# generate a sequence by sampling from the predicted distribution at each step
print()
print(generate(w2c_rnn, word_vectors, index_to_vocab, seed_word='Machine', num_words=10, sample=True))

next word after 'Machine': learning

Machine automate (ML) is a branch of Artificial learning continues to


### 4.2 Glove - Pre-trained embeddings (Google-News word2vec)

Swap our small in-house vectors for Google's pre-trained 300-dimensional
word2vec vectors (trained on ~100B words of Google News). Rich, general-purpose
embeddings usually train faster and generalize better than the tiny model above.

In [8]:
# pre trained glove by google news dataset (300d vectors, 3 million words)
import gensim.downloader as api
glove = api.load("word2vec-google-news-300")

In [9]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(words, T_x=5, is_char=False, word_vectors=glove)
print(f"input_seqs shape: {input_seqs.shape} target_seqs shape: {target_seqs.shape} (vector_size, m, T_x)")

glove_rnn = RNN(input_seqs, target_seqs, n_a=100, learning_rate=0.001, iterations=2000,
                task="classification")
glove_rnn.train()

input_seqs shape: (300, 72, 5) target_seqs shape: (67, 72, 5) (vector_size, m, T_x)
Iteration: 0, Loss: 21.02409792226509
Iteration: 100, Loss: 17.81826332418981
Iteration: 200, Loss: 4.547109299295624
Iteration: 300, Loss: 1.5212635919622215
Iteration: 400, Loss: 0.9084196146144533
Iteration: 500, Loss: 0.6516623818352747
Iteration: 600, Loss: 0.5141809268905521
Iteration: 700, Loss: 0.43285272565558486
Iteration: 800, Loss: 0.38065432701122187
Iteration: 900, Loss: 0.34491912814603787
Iteration: 1000, Loss: 0.31922126255001854
Iteration: 1100, Loss: 0.300018330833258
Iteration: 1200, Loss: 0.2852217973049268
Iteration: 1300, Loss: 0.2735319696929655
Iteration: 1400, Loss: 0.26410323002970176
Iteration: 1500, Loss: 0.25636449324715177
Iteration: 1600, Loss: 0.24991770636646532
Iteration: 1700, Loss: 0.2444778302938795
Iteration: 1800, Loss: 0.23983594621184473
Iteration: 1900, Loss: 0.23583579638186475


In [10]:
# single next-word prediction (most likely word)
print("next word after 'Machine':", predict_next(glove_rnn, glove, index_to_vocab, 'Machine'))

# generate a sequence by sampling from the predicted distribution at each step
print()
print(generate(glove_rnn, glove, index_to_vocab, seed_word='Machine', num_words=10, sample=True))

next word after 'Machine': learning

Machine learning is widely used in various applications such as recommendation


### 4.3 FastText embeddings

FastText builds word vectors from character n-grams, so it can embed even rare
or out-of-vocabulary words. Here we train it on our own sentences.

In [11]:
from gensim.models import FastText

model = FastText(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
fasttext_vectors = model.wv

In [12]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(words, T_x=5, is_char=False, word_vectors=fasttext_vectors)
print(f"input_seqs shape: {input_seqs.shape} target_seqs shape: {target_seqs.shape} (vector_size, m, T_x)")

fasttext_rnn = RNN(input_seqs, target_seqs, n_a=50, learning_rate=0.01193, iterations=2000,
                   task="classification")
fasttext_rnn.train()

input_seqs shape: (100, 102, 5) target_seqs shape: (88, 102, 5) (vector_size, m, T_x)
Iteration: 0, Loss: 22.386679760150194
Iteration: 100, Loss: 21.75093151913289
Iteration: 200, Loss: 21.736251170464158
Iteration: 300, Loss: 21.724288511874118
Iteration: 400, Loss: 21.718098139607665
Iteration: 500, Loss: 21.711537897914344
Iteration: 600, Loss: 21.70762717980347
Iteration: 700, Loss: 21.7022959984859
Iteration: 800, Loss: 21.69969717165455
Iteration: 900, Loss: 21.69381460270545
Iteration: 1000, Loss: 21.689478141425226
Iteration: 1100, Loss: 21.6799419289197
Iteration: 1200, Loss: 21.659046307834036
Iteration: 1300, Loss: 21.583659938527088
Iteration: 1400, Loss: 21.39106133350092
Iteration: 1500, Loss: 19.227553955128414
Iteration: 1600, Loss: 18.519296858060237
Iteration: 1700, Loss: 17.855939555144268
Iteration: 1800, Loss: 17.033677627476397
Iteration: 1900, Loss: 13.580854252999806


In [13]:
# single next-word prediction (most likely word)
print("next word after 'Machine':", predict_next(fasttext_rnn, fasttext_vectors, index_to_vocab, 'Machine'))

# generate a sequence by sampling from the predicted distribution at each step
print()
print(generate(fasttext_rnn, fasttext_vectors, index_to_vocab, seed_word='Machine', num_words=10, sample=True))

next word after 'Machine': and

Machine as reinforcement grow, machine Intelligence learning, Artificial detection, (AI) to


### 4.4 BERT embeddings

Use a pre-trained BERT model to produce a contextual `[CLS]` embedding for each
word, then feed those 768-dimensional vectors to the RNN.

In [14]:
from transformers import BertTokenizer, BertModel
import torch
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

bert_word_to_vec = {}
for word in words:
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state[:,0,:]
    bert_word_to_vec[word] = embeddings.squeeze().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(words, T_x=5, is_char=False, word_vectors=bert_word_to_vec)
print(f"input_seqs shape: {input_seqs.shape} target_seqs shape: {target_seqs.shape} (vector_size, m, T_x)")

bert_rnn = RNN(input_seqs, target_seqs, n_a=50, learning_rate=0.01193, iterations=2000,
               task="classification")
bert_rnn.train()

input_seqs shape: (768, 102, 5) target_seqs shape: (88, 102, 5) (vector_size, m, T_x)
Iteration: 0, Loss: 22.389862923194364
Iteration: 100, Loss: 8.252428455450083
Iteration: 200, Loss: 0.6923491944007096
Iteration: 300, Loss: 0.4360399469462813
Iteration: 400, Loss: 0.4185452967008284
Iteration: 500, Loss: 0.40577481376000074
Iteration: 600, Loss: 0.4007212657514265
Iteration: 700, Loss: 0.39855892182200636
Iteration: 800, Loss: 0.3957489047032326
Iteration: 900, Loss: 0.393573312129074
Iteration: 1000, Loss: 0.39291806962775966
Iteration: 1100, Loss: 0.3917166042586116
Iteration: 1200, Loss: 0.3904342837008754
Iteration: 1300, Loss: 0.38997020945256067
Iteration: 1400, Loss: 0.3892303922589826
Iteration: 1500, Loss: 0.3886476686387093
Iteration: 1600, Loss: 0.3883622363437358
Iteration: 1700, Loss: 0.38813656723513174
Iteration: 1800, Loss: 0.38769649663792094
Iteration: 1900, Loss: 0.3874005093767053


In [16]:
# single next-word prediction (most likely word)
print("next word after 'Machine':", predict_next(bert_rnn, bert_word_to_vec, index_to_vocab, 'Machine'))

# generate a sequence by sampling from the predicted distribution at each step
print()
print(generate(bert_rnn, bert_word_to_vec, index_to_vocab, seed_word='Machine', num_words=10, sample=True))

next word after 'Machine': learning

Machine learning plays an increasingly important role in helping organizations automate
